# Melbourne Solar Project - Part 3: Economic & Environmental Analysis

**Teammate:** Jawaharsriraam Sathya Rajendran  
**Date:** May 2026  
**Focus:** Cost analysis, savings projections, and CO2 reduction calculations

---

## Objective
Analyze the economic viability and environmental benefits of solar installations across Melbourne buildings.


In [120]:
# Import required libraries
%pip install matplotlib seaborn folium geopandas
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"GeoPandas version: {gpd.__version__}")
print(f"NumPy version: {np.__version__}")


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ All libraries imported successfully!
Pandas version: 3.0.2
GeoPandas version: 1.1.3
NumPy version: 2.4.4


In [121]:
# Define paths
BASE_DIR = "/Users/jawaharsr/Downloads/MOPProject"  # Adjust if notebook is in different location
BASE_DIR = Path(BASE_DIR)
RAW_DATA = BASE_DIR / 'RAW_DATA'
PROCESSED_DATA = BASE_DIR / 'Processed'

print("=" * 80)
print("LOADING DATA")
print("=" * 80)

# 1. Load Building Footprints
print("\n📂 Loading building footprints...")
buildings = gpd.read_file(RAW_DATA / 'melbourne_buildings_inner.geojson')
print(f"   ✓ Loaded {len(buildings):,} buildings")
print(f"   ✓ CRS: {buildings.crs}")
print(f"   ✓ Columns: {list(buildings.columns[:5])}...")

# 2. Load Solar Irradiance Data
print("\n☀️ Loading solar irradiance data...")
solar = pd.read_csv(RAW_DATA / 'solar_irradiance_melbourne.csv')
print(f"   ✓ Loaded solar data for {len(solar)} postcodes")

# 3. Load Electricity Pricing
print("\n💰 Loading electricity pricing...")
pricing = pd.read_csv(RAW_DATA / 'electricity_pricing.csv')
print(f"   ✓ Loaded {len(pricing)} pricing rates")
display(pricing)

print("\n✅ All data loaded successfully!")

LOADING DATA

📂 Loading building footprints...
   ✓ Loaded 41,701 buildings
   ✓ CRS: EPSG:4326
   ✓ Columns: ['geo_point_2d', 'objectid', 'structure_id', 'footprint_type', 'roof_type']...

☀️ Loading solar irradiance data...
   ✓ Loaded solar data for 313 postcodes

💰 Loading electricity pricing...
   ✓ Loaded 3 pricing rates


,rate_type,rate_aud,description
0,consumption,0.29,Average consumption rate (VDO)
1,feed_in_tariff,0.05,Minimum feed-in tariff
2,daily_supply,1.05,Daily supply charge



✅ All data loaded successfully!


---

## Step 1: Load Data from Previous Parts


In [ ]:
# Load the processed data
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load buildings with system sizing
buildings = gpd.read_file(buildings_path)

print(f"Loaded {len(buildings):,} buildings with system data")
print(f"Average system size: {buildings['max_system_size_kw'].mean():.2f} kW")


# ============================================================================
# ADDITIONAL SECTION 3D: CALCULATE COSTS AND SAVINGS
# ============================================================================

In [131]:
print("\n" + "=" * 80)
print("ECONOMIC ANALYSIS - COSTS & SAVINGS")
print("=" * 80)

# Installation costs ($/kW) with economies of scale
def calculate_cost_per_kw(system_kw):
    if system_kw <= 6.6:
        return 1800
    elif system_kw <= 10:
        return 1600
    elif system_kw <= 30:
        return 1400
    elif system_kw <= 100:
        return 1200
    else:
        return 1000

print("\n💵 Calculating installation costs...")
buildings['cost_per_kw'] = buildings['recommended_system_kw'].apply(calculate_cost_per_kw)
buildings['installation_cost_aud'] = buildings['recommended_system_kw'] * buildings['cost_per_kw']

# Solar Victoria rebates
def calculate_rebate(category, system_kw):
    if category.startswith('Residential') and system_kw <= 13.5:
        return min(1400, system_kw * 100)
    return 0

print("\n🎁 Calculating rebates...")
buildings['rebate_aud'] = buildings.apply(
    lambda row: calculate_rebate(row['building_category'], row['recommended_system_kw']),
    axis=1
)

buildings['out_of_pocket_cost_aud'] = buildings['installation_cost_aud'] - buildings['rebate_aud']

# Annual savings
ELECTRICITY_RATE = 0.2934  # $/kWh
FEED_IN_TARIFF = 0.053     # $/kWh

def get_self_consumption_rate(category):
    if 'Residential' in category:
        return 0.30
    else:
        return 0.60

print("\n💰 Calculating annual savings...")
buildings['self_consumption_rate'] = buildings['building_category'].apply(get_self_consumption_rate)
buildings['self_consumed_kwh'] = buildings['annual_generation_kwh'] * buildings['self_consumption_rate']
buildings['exported_kwh'] = buildings['annual_generation_kwh'] * (1 - buildings['self_consumption_rate'])

buildings['annual_savings_aud'] = (
    buildings['self_consumed_kwh'] * ELECTRICITY_RATE +
    buildings['exported_kwh'] * FEED_IN_TARIFF
)

# Payback period
buildings['payback_years'] = buildings['out_of_pocket_cost_aud'] / buildings['annual_savings_aud']
buildings['payback_years'] = buildings['payback_years'].replace([np.inf, -np.inf], np.nan)

# Lifetime value
buildings['lifetime_savings_25yr'] = (buildings['annual_savings_aud'] * 25) - buildings['out_of_pocket_cost_aud']

print("\n📊 Economic Summary:")
print(f"   Median installation cost: ${buildings['installation_cost_aud'].median():,.0f}")
print(f"   Median annual savings: ${buildings['annual_savings_aud'].median():.0f}")
print(f"   Median payback: {buildings['payback_years'].median():.1f} years")
print(f"   Total investment required: ${buildings['out_of_pocket_cost_aud'].sum()/1_000_000:.1f}M")


ECONOMIC ANALYSIS - COSTS & SAVINGS

💵 Calculating installation costs...

🎁 Calculating rebates...

💰 Calculating annual savings...

📊 Economic Summary:
   Median installation cost: $11,700
   Median annual savings: $997
   Median payback: 11.1 years
   Total investment required: $1179.6M


# ============================================================================
# ADDITIONAL SECTION 3E: CALCULATE CO2 REDUCTION
# ============================================================================

In [132]:
print("\n" + "=" * 80)
print("ENVIRONMENTAL IMPACT - CO2 REDUCTION")
print("=" * 80)

# Victoria grid carbon intensity
GRID_CARBON_INTENSITY = 0.81  # kg CO2/kWh

print(f"\n🏭 Grid carbon intensity: {GRID_CARBON_INTENSITY} kg CO2/kWh (Victoria 2024)")

# Calculate CO2 reduction
print("\n🌱 Calculating CO2 reduction...")
buildings['annual_co2_reduction_kg'] = buildings['annual_generation_kwh'] * GRID_CARBON_INTENSITY
buildings['annual_co2_reduction_tonnes'] = buildings['annual_co2_reduction_kg'] / 1000
buildings['lifetime_co2_reduction_tonnes'] = buildings['annual_co2_reduction_tonnes'] * 25

# Trees equivalent
buildings['trees_equivalent'] = (buildings['annual_co2_reduction_kg'] / 20).round(0)

print("\n📊 Environmental Impact:")
print(f"   Total annual CO2 reduction: {buildings['annual_co2_reduction_tonnes'].sum():,.0f} tonnes")
print(f"   Equivalent trees: {buildings['trees_equivalent'].sum():,.0f}")
print(f"   Equivalent cars removed: {buildings['annual_co2_reduction_tonnes'].sum()/4.6:,.0f}")


ENVIRONMENTAL IMPACT - CO2 REDUCTION

🏭 Grid carbon intensity: 0.81 kg CO2/kWh (Victoria 2024)

🌱 Calculating CO2 reduction...

📊 Environmental Impact:
   Total annual CO2 reduction: 976,616 tonnes
   Equivalent trees: 48,831,265
   Equivalent cars removed: 212,308


---

## Additional Analysis by Teammate 1 (Continued)

### Payback Period Analysis
Detailed analysis of investment return periods across different building types.


In [ ]:
# Additional analysis - Payback period breakdown
print("\n" + "=" * 80)
print("PAYBACK PERIOD ANALYSIS (Teammate 1 Extra)")
print("=" * 80)

# Calculate payback period
buildings['payback_years'] = buildings['installation_cost'] / buildings['annual_savings'].replace(0, np.nan)

# Filter realistic payback periods
buildings_payback = buildings[buildings['payback_years'] < 30].copy()

print("\n💰 Payback Period Statistics:")
print(f"Mean payback: {buildings_payback['payback_years'].mean():.1f} years")
print(f"Median payback: {buildings_payback['payback_years'].median():.1f} years")
print(f"Best payback: {buildings_payback['payback_years'].min():.1f} years")

# By building type
print("\n📊 Payback by Building Type:")
for btype in ['Residential', 'Commercial', 'Industrial']:
    subset = buildings_payback[buildings_payback['building_type'] == btype]
    if len(subset) > 0:
        print(f"  {btype}: {subset['payback_years'].mean():.1f} years (avg)")

# Net Present Value (NPV) analysis
DISCOUNT_RATE = 0.05  # 5% discount rate
SYSTEM_LIFETIME = 25  # years

def calculate_npv(row):
    annual_cf = row['annual_savings']
    initial_cost = row['installation_cost']
    npv = -initial_cost
    for year in range(1, SYSTEM_LIFETIME + 1):
        npv += annual_cf / ((1 + DISCOUNT_RATE) ** year)
    return npv

buildings['npv_25years'] = buildings.apply(calculate_npv, axis=1)
buildings['roi_percent'] = (buildings['npv_25years'] / buildings['installation_cost']) * 100

print(f"\n📈 Financial Metrics (25-year lifetime):")
print(f"Total NPV: ${buildings['npv_25years'].sum():,.0f}")
print(f"Average ROI: {buildings['roi_percent'].mean():.1f}%")
print(f"Buildings with positive NPV: {(buildings['npv_25years'] > 0).sum():,} ({(buildings['npv_25years'] > 0).mean()*100:.1f}%)")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Payback distribution
axes[0, 0].hist(buildings_payback['payback_years'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 0].set_xlabel('Payback Period (years)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Payback Period Distribution')
axes[0, 0].axvline(buildings_payback['payback_years'].median(), color='red', linestyle='--', label='Median')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# ROI by building type
roi_by_type = buildings.groupby('building_type')['roi_percent'].mean()
axes[0, 1].bar(roi_by_type.index, roi_by_type.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 1].set_ylabel('Average ROI (%)')
axes[0, 1].set_title('Return on Investment by Building Type')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# NPV vs System Size
axes[1, 0].scatter(buildings['max_system_size_kw'], buildings['npv_25years'], alpha=0.3, c=buildings['annual_savings'], cmap='viridis')
axes[1, 0].set_xlabel('System Size (kW)')
axes[1, 0].set_ylabel('Net Present Value ($)')
axes[1, 0].set_title('NPV vs System Size')
axes[1, 0].grid(True, alpha=0.3)
cbar = plt.colorbar(axes[1, 0].collections[0], ax=axes[1, 0])
cbar.set_label('Annual Savings ($)')

# Cumulative savings over time
years = np.arange(0, 26)
median_system = buildings[buildings['max_system_size_kw'] > 0].iloc[len(buildings)//2]
cumulative_savings = [-median_system['installation_cost']]
for year in range(1, 26):
    cumulative_savings.append(cumulative_savings[-1] + median_system['annual_savings'])

axes[1, 1].plot(years, cumulative_savings, linewidth=2, color='green')
axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].fill_between(years, 0, cumulative_savings, where=[x > 0 for x in cumulative_savings], 
                        alpha=0.3, color='green', label='Profit')
axes[1, 1].fill_between(years, cumulative_savings, 0, where=[x < 0 for x in cumulative_savings], 
                        alpha=0.3, color='red', label='Investment Period')
axes[1, 1].set_xlabel('Years')
axes[1, 1].set_ylabel('Cumulative Cash Flow ($)')
axes[1, 1].set_title('Median System: Cumulative Savings Over Time')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Payback period analysis complete!")


---

## Key Findings from Part 3
1. Calculated installation costs with economies of scale
2. Projected annual savings based on energy generation
3. Estimated CO2 emissions reduction potential
4. Analyzed payback periods and NPV for financial viability
5. Most buildings show positive ROI within system lifetime
6. Ready for opportunity scoring in Part 4
